<div align="center">
  <img src="assets/fleetguard-logo.svg" alt="FleetGuard AI" width="360">
</div>

# 01 · Dataset overview
**Goal:** identify the cohort, exposure and gaps before looking at signal behaviour.

## Review at a glance

**Source and feature validation passed.** The reviewed snapshot is the seed-42,
10-second `portfolio-v1` cohort generated with package 0.4.0 and mapped railway
geometry. Findings below describe this saved snapshot; refresh them if the
configuration or dataset changes.

| Fleet-wide inventory | Result |
| --- | ---: |
| Wagons / bogies / axles / wheels | 48 / 96 / 192 / 384 |
| Wagons per train / validation / test split | 16 / 16 / 16 |
| Scheduled / received / intentionally missing wagon reports | 78,954 / 75,277 / 3,677 |
| One-minute feature rows across all component levels | 197,385 |

Signal exploration below uses **training data**. Validation and test are included
only in the inventory and integrity checks. This notebook establishes coverage
and reporting behaviour; signal quality and modelling suitability need the next
EDA sessions.

In [1]:
import os
import sys
from pathlib import Path

# Works when Jupyter starts from the repository root or from nbs.
HELPERS = Path.cwd() if (Path.cwd() / "fleetguard_eda.py").exists() else Path.cwd() / "nbs"
sys.path.insert(0, str(HELPERS))
import fleetguard_eda as eda
import panel as pn

eda.setup()
ROOT = eda.project_root()
SOURCE = Path(os.environ.get("FLEETGUARD_EDA_SOURCE", ROOT / "data/generated/portfolio-v1"))
FEATURES = Path(os.environ.get("FLEETGUARD_EDA_FEATURES", ROOT / "data/features/portfolio-v1"))
SPLIT = "train"  # change to validation to check findings; test is blocked for exploration
ws = eda.Workspace(SOURCE, FEATURES)
pn.pane.Markdown(
    f"**Source:** `{SOURCE}`  ·  **Features:** `{FEATURES}`  ·  **Explore:** `{SPLIT}`\n\n"
    "Maps need internet. Linked controls need a running, trusted notebook kernel."
)

Markdown(str)

### 1 · Confirm the source
These checks scan both saved datasets. Test is used only for inventory/coverage.

In [2]:
CHECK_SAVED_FILES = True
checks = (
    ws.validate()
    if CHECK_SAVED_FILES
    else {"note": "Validation skipped; run CLI validators first."}
)
checks

{'portfolio': 'passed', 'features': 'passed'}

**Result:** portfolio and feature validation both passed. The matching manifests
and audit were also reviewed. This confirms the saved-data checks, not model accuracy.

### 2 · Coverage and unique identities
Open each tab. Counts use complete parent keys, not just inner/outer or left/right.

In [3]:
eda.overview_views(ws)

Tabs
    [0] Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')
    [1] Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')
    [2] Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')
    [3] Tabulator(disabled=True, page_size=10, pagination='local', show_index=False, sizing_mode='stretch_width', value=        split  ...)
    [4] Tabulator(disabled=True, page_size=10, pagination='local', show_index=False, sizing_mode='stretch_width', value=         split  ...)

**Coverage finding:** each split has 16 distinct wagons: 13 fault variants and
3 healthy controls. All four routes occur in each split, but route counts and duty
lengths differ. In training, duties range from **148 to 533 minutes**; longer
journeys contribute more reports. This deliberately fault-rich cohort does not
represent real fleet failure prevalence.

Training route assignments: Avonmouth–Cardiff **4**, Cardiff–Swansea **5**,
London–Avonmouth **4**, London–Swansea **3**.

### Component reporting frequency
Actual received-report counts use full component keys. Train/validation only.

In [4]:
eda.identity_frequency(ws, SPLIT)

Tabulator(disabled=True, page_size=10, pagination='local', show_index=False, sizing_mode='stretch_width', value=        asset_id  ...)

**Identity check:** the training split contains **16 wagons → 32 bogies →
64 axles → 128 wheels**. Its **24,507 received wagon reports** correspond to
49,014 bogie, 98,028 axle and 196,056 wheel observations. These are linked levels
of the same snapshots, not independent examples. Full parent keys are required.

### 3 · When did reports disappear?
White cells outside a duty are not outages. Completeness uses expected report slots.

In [5]:
eda.pane(eda.completeness_view(ws, SPLIT))

Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')

**Completeness finding:** training contains **4,301 wagon-minute windows**:
**4,083 complete**, **3 partial** and **215 silent**. Of **25,806 scheduled reports**,
**24,507 were received (94.97%)** and **1,299 are intentionally absent**.
Blank cells beyond a wagon's journey are outside its observation horizon, not
outages. Silent windows must stay distinct from received reports with unavailable sensors.

### 4 · Explore one wagon on the map
Markers are sampled; heat layers use all received wagon reports. No wheel duplication.

In [6]:
wagons = ws.load("wagon", SPLIT)
asset_picker = pn.widgets.Select(name="Wagon", options=["All", *sorted(wagons.asset_id.unique())])


def route_view(asset):
    selected = eda.choose(wagons, asset=asset)
    return pn.pane.HTML(
        eda.folium_map(selected)._repr_html_(), height=620, sizing_mode="stretch_width"
    )


pn.Column(asset_picker, pn.bind(route_view, asset_picker))

Column
    [0] Select(label='Wagon', name='Wagon', options=['All', 'FG-WGN-0002', ...], value='All')
    [1] ParamFunction(function, _pane=HTML, defer_load=False)

**Map reading:** select one wagon to inspect its duty without overlapping other
wagons. The line links received positions; a segment spanning an outage is not
continuous observed tracking. Aerial/street/topographic layers provide context;
clusters show sampled reports and heat layers show reporting density, not fault risk.

The time slider replays received-report density in **30-minute bins**. It is not
a 10-second position replay or a filter for all map layers. Marker sampling and
coordinate rounding in heat layers limit the spatial detail. Python-backed wagon
selection needs the running notebook; exported HTML retains its saved selection.

### 5 · Outage replay
Expected missing slots are synthetic truth, not observations of the controller during silence.

In [7]:
eda.pane(eda.outage_view(ws, SPLIT))

Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')

**Outage finding:** the saved replay confirms **1,299 missing scheduled reports**
on the three training assets below. Missing-slot positions on the time plot are
known synthetic truth, not measurements transmitted while the controller was silent.

| Training wagon | Assigned scenario | Missing reports | Silent minutes | Partial minutes |
| --- | --- | ---: | ---: | ---: |
| FG-WGN-0004 | Premature battery depletion | 807 | 134 | 1 |
| FG-WGN-0021 | Persistent controller supply failure | 420 | 70 | 0 |
| FG-WGN-0039 | Bounded controller outage with recovery | 72 | 11 | 2 |

The battery case has two distinct gaps, so these three assets produce four outage
intervals. Recovery is visible between its gaps; bounded controller reporting also
resumes. The persistent controller case remains silent to the journey horizon.
No separate communications-network fault is modelled.

### 6 · Decisions and next session

| Decision | Evidence / action |
| --- | --- |
| Keep this dataset for EDA | Source/feature validation passed and audit counts reconcile. No regeneration is needed for this review. |
| Preserve component identities | Use wagon + bogie + axle + wheel keys as appropriate; component rows share parent snapshots. |
| Account for unequal exposure | Routes have 3–5 training wagons each, and duty lengths range from 148–533 minutes. Compare rates with their exposure, not raw counts alone. |
| Preserve intentional silence | Retain expected slots and completeness. Do not replace absent reports with zero-valued measurements. |
| Keep truth out of predictors | Scenario assignments and missing-report truth explain the simulator; they are evaluation evidence, not production observations. |
| Reserve test data | Use training for exploration, validation for checking choices, and test for final evaluation. |

**Next: `02_healthy_signal_behaviour.ipynb`.** Start with healthy-control wagons,
review all signals at all four levels, and examine operating-state relationships
before selecting any model inputs. Dataset consistency is established; model
performance and real-world generalisation have not yet been assessed.